# Play-by-Play Event Exploration

## NBA Player Impact and Lineup Optimization Dashboard

This notebook inspects the structure and logic of the `play_by_play` table, which serves as the event-level foundation of the project.

The goal is to understand how game events are stored, ordered, and attributed to teams and players, so that substitutions, possessions, and scoring progression can be reconstructed accurately in later notebooks.

## Why this step matters

Lineup analytics depends on correctly reconstructing who was on the floor, when substitutions occurred, and how score and possessions evolved over time.

Before building stint-level datasets, we need to validate that the play-by-play table contains enough information to support:

- chronological event ordering
- game and period segmentation
- player and team attribution
- substitution detection
- score progression
- event classification for later possession logic

This notebook focuses on understanding those mechanics in a structured and reproducible way.

## Objectives

By the end of this notebook, we want to:

- inspect the schema of the `play_by_play` table
- identify the columns needed for event sequencing
- examine how scoring and substitutions are represented
- explore event descriptions and event types
- test a few single-game extracts to verify event flow
- define the fields that will be required for stint construction in Notebook 03

In [2]:
import sqlite3
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)

DB_PATH = "../data/raw/nba.sqlite"
conn = sqlite3.connect(DB_PATH)

## Inspect the play-by-play schema

We begin by reviewing the column structure of the `play_by_play` table.

The main purpose of this section is to identify the columns that may contain:

- game identifiers
- event sequence
- period and game clock
- player and team IDs
- event descriptions
- score updates

In [3]:
pbp_schema = pd.read_sql("PRAGMA table_info(play_by_play);", conn)
pbp_schema

,cid,name,type,notnull,dflt_value,pk
0,0,game_id,TEXT,0,None,0
1,1,eventnum,INTEGER,0,None,0
2,2,eventmsgtype,INTEGER,0,None,0
3,3,eventmsgactiontype,INTEGER,0,None,0
4,4,period,INTEGER,0,None,0
5,5,wctimestring,TEXT,0,None,0
6,6,pctimestring,TEXT,0,None,0
7,7,homedescription,TEXT,0,None,0
8,8,neutraldescription,TEXT,0,None,0
9,9,visitordescription,TEXT,0,None,0


## Initial schema observations

The `play_by_play` table contains the key fields required for downstream event reconstruction:

- `game_id` for identifying games
- `eventnum` for ordering events
- `eventmsgtype` and `eventmsgactiontype` for classifying actions
- `period` and `pctimestring` for game chronology
- `score` and `scoremargin` for tracking scoring progression
- `homedescription`, `visitordescription`, and `neutraldescription` for event text
- `player1_*`, `player2_*`, and `player3_*` for player and team attribution

This is a strong foundation for constructing possession- and stint-level analytics.

## Preview the raw play-by-play table 

A global preview helps verify how these fields appear in practice before focusing on one game

In [4]:
pbp_preview = pd.read_sql("SELECT * FROM play_by_play LIMIT 10;", conn)
pbp_preview

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,score,scoremargin,person1type,player1_id,player1_name,player1_team_id,player1_team_city,player1_team_nickname,player1_team_abbreviation,person2type,player2_id,player2_name,player2_team_id,player2_team_city,player2_team_nickname,player2_team_abbreviation,person3type,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag
0,0029600012,0,12,0,1,14:43 PM,12:00,None,Start of 1st Period (14:43 PM EST),None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
1,0029600012,2,10,0,1,14:50 PM,12:00,Jump Ball O'Neal vs. Kleine: Tip to Cassell,None,None,None,None,4.0,406,Shaquille O'Neal,1610612747.0,Los Angeles,Lakers,LAL,5.0,170,Joe Kleine,1610612756.0,Phoenix,Suns,PHX,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0
2,0029600012,3,2,1,1,14:51 PM,11:45,None,None,MISS Cassell 15' Jump Shot,None,None,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
3,0029600012,4,4,0,1,14:51 PM,11:43,O'Neal REBOUND (Off:0 Def:1),None,None,None,None,4.0,406,Shaquille O'Neal,1610612747.0,Los Angeles,Lakers,LAL,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
4,0029600012,5,2,1,1,14:51 PM,11:29,MISS Ceballos 26' 3PT Jump Shot,None,None,None,None,4.0,76,Cedric Ceballos,1610612747.0,Los Angeles,Lakers,LAL,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
5,0029600012,6,4,0,1,14:51 PM,11:27,None,None,Cassell REBOUND (Off:0 Def:1),None,None,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
6,0029600012,7,6,1,1,14:51 PM,11:14,Van Exel P.FOUL (P1.T1),None,None,None,None,4.0,89,Nick Van Exel,1610612747.0,Los Angeles,Lakers,LAL,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
7,0029600012,8,5,1,1,14:52 PM,11:08,None,None,Cassell Bad Pass Turnover (P1.T1),None,None,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
8,0029600012,9,2,5,1,14:52 PM,10:49,MISS Ceballos 1' Layup,None,Horry BLOCK (1 BLK),None,None,4.0,76,Cedric Ceballos,1610612747.0,Los Angeles,Lakers,LAL,0.0,0,None,None,None,None,None,5.0,109,Robert Horry,1610612756.0,Phoenix,Suns,PHX,0
9,0029600012,10,4,0,1,14:52 PM,10:49,LAKERS Rebound,None,None,None,None,2.0,1610612747,None,None,None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0


## Select a multi-game sample for event exploration

The original prototype inspected one game to understand play-by-play structure. Since the project is now moving toward multi-game modeling, we select a manageable sample of games for exploration.

This allows us to verify that event ordering, substitutions, score progression, and player/team references behave consistently across multiple games before scaling the stint reconstruction pipeline.

## Filter to regular season and playoff games

To reduce noise from preseason rotations, the multi-game sample is restricted to regular season and playoff games.

In this dataset, game identifiers can be filtered by prefix:

- `002` = regular season
- `004` = playoffs

This makes the sample more relevant for player impact, lineup modeling, and coaching-oriented analysis.

In [5]:
sample_games = pd.read_sql(
    """
    SELECT DISTINCT game_id
    FROM play_by_play
    WHERE game_id LIKE '002%' 
       OR game_id LIKE '004%'
    ORDER BY game_id
    LIMIT 500;
    """,
    conn
)

sample_games.head()

,game_id
0,0020000001
1,0020000002
2,0020000003
3,0020000004
4,0020000005


In [6]:
sample_game_ids = sample_games["game_id"].tolist()

placeholders = ",".join(["?"] * len(sample_game_ids))

pbp_sample = pd.read_sql(
    f"""
    SELECT *
    FROM play_by_play
    WHERE game_id IN ({placeholders})
    ORDER BY game_id, period, eventnum;
    """,
    conn,
    params=sample_game_ids
)

pbp_sample.shape

(227577, 34)

In [7]:
pbp_sample["game_id"].str[:3].value_counts()

002    227577
Name: game_id, dtype: int64

In [8]:
pbp_sample["game_id"].nunique(), pbp_sample.shape

(500, (227577, 34))

## Select one representative game for detailed inspection

Although the notebook now uses a multi-game sample, detailed event inspection is still easier at the single-game level.

We select one game from the sample to visually inspect event sequencing, substitutions, scoring updates, and player roles.

In [9]:
sample_game_id = sample_game_ids[0]

sample_game = (
    pbp_sample[pbp_sample["game_id"] == sample_game_id]
    .sort_values(["period", "eventnum"])
    .copy()
)

sample_game_id, sample_game.shape

('0020000001', (429, 34))

In [10]:
sample_game.head(20)

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,score,scoremargin,person1type,player1_id,player1_name,player1_team_id,player1_team_city,player1_team_nickname,player1_team_abbreviation,person2type,player2_id,player2_name,player2_team_id,player2_team_city,player2_team_nickname,player2_team_abbreviation,person3type,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag
0,0020000001,0,12,0,1,12:13 PM,12:00,None,Start of 1st Period (12:13 PM EST),None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
1,0020000001,1,10,0,1,12:13 PM,12:00,Jump Ball Camby vs. Ratliff: Tip to Houston,None,None,None,None,4.0,948,Marcus Camby,1610612752.0,New York,Knicks,NYK,5.0,689,Theo Ratliff,1610612755.0,Philadelphia,76ers,PHI,4.0,275,Allan Houston,1610612752.0,New York,Knicks,NYK,0
2,0020000001,2,2,1,1,12:14 PM,11:41,MISS Sprewell 6' Jump Shot,None,Ratliff BLOCK (1 BLK),None,None,4.0,84,Latrell Sprewell,1610612752.0,New York,Knicks,NYK,0.0,0,None,None,None,None,None,5.0,689,Theo Ratliff,1610612755.0,Philadelphia,76ers,PHI,0
3,0020000001,3,4,0,1,12:14 PM,11:40,None,None,76ers Rebound,None,None,3.0,1610612755,None,None,None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
4,0020000001,4,6,2,1,12:14 PM,11:29,Camby S.FOUL (P1.T1),None,None,None,None,4.0,948,Marcus Camby,1610612752.0,New York,Knicks,NYK,0.0,0,None,None,None,None,None,1.0,0,None,None,None,None,None,0
5,0020000001,5,3,11,1,12:14 PM,11:29,None,None,Ratliff Free Throw 1 of 2 (1 PTS),1 - 0,-1,5.0,689,Theo Ratliff,1610612755.0,Philadelphia,76ers,PHI,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
6,0020000001,6,3,12,1,12:15 PM,11:29,None,None,MISS Ratliff Free Throw 2 of 2,None,None,5.0,689,Theo Ratliff,1610612755.0,Philadelphia,76ers,PHI,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
7,0020000001,7,4,0,1,12:15 PM,11:28,Ward REBOUND (Off:0 Def:1),None,None,None,None,4.0,369,Charlie Ward,1610612752.0,New York,Knicks,NYK,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
8,0020000001,8,6,2,1,12:15 PM,11:18,None,None,Ratliff S.FOUL (P1.T1),None,None,5.0,689,Theo Ratliff,1610612755.0,Philadelphia,76ers,PHI,0.0,0,None,None,None,None,None,1.0,0,None,None,None,None,None,0
9,0020000001,9,3,11,1,12:16 PM,11:18,Camby Free Throw 1 of 2 (1 PTS),None,None,1 - 1,TIE,4.0,948,Marcus Camby,1610612752.0,New York,Knicks,NYK,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0


##  Build a focused event view

This focused view makes it easier to inspect:

- event sequence
- game clock
- event descriptions
- score changes
- player references

In [11]:
core_cols = [
    "game_id",
    "eventnum",
    "period",
    "pctimestring",
    "eventmsgtype",
    "eventmsgactiontype",
    "score",
    "scoremargin",
    "homedescription",
    "visitordescription",
    "neutraldescription",
    "player1_id",
    "player1_name",
    "player1_team_id",
    "player2_id",
    "player2_name",
    "player2_team_id",
    "player3_id",
    "player3_name",
    "player3_team_id",
]

sample_game_core = sample_game[core_cols].copy()
sample_game_core.head(30)

,game_id,eventnum,period,pctimestring,eventmsgtype,eventmsgactiontype,score,scoremargin,homedescription,visitordescription,neutraldescription,player1_id,player1_name,player1_team_id,player2_id,player2_name,player2_team_id,player3_id,player3_name,player3_team_id
0,0020000001,0,1,12:00,12,0,None,None,None,None,Start of 1st Period (12:13 PM EST),0,None,None,0,None,None,0,None,None
1,0020000001,1,1,12:00,10,0,None,None,Jump Ball Camby vs. Ratliff: Tip to Houston,None,None,948,Marcus Camby,1610612752.0,689,Theo Ratliff,1610612755.0,275,Allan Houston,1610612752.0
2,0020000001,2,1,11:41,2,1,None,None,MISS Sprewell 6' Jump Shot,Ratliff BLOCK (1 BLK),None,84,Latrell Sprewell,1610612752.0,0,None,None,689,Theo Ratliff,1610612755.0
3,0020000001,3,1,11:40,4,0,None,None,None,76ers Rebound,None,1610612755,None,None,0,None,None,0,None,None
4,0020000001,4,1,11:29,6,2,None,None,Camby S.FOUL (P1.T1),None,None,948,Marcus Camby,1610612752.0,0,None,None,0,None,None
5,0020000001,5,1,11:29,3,11,1 - 0,-1,None,Ratliff Free Throw 1 of 2 (1 PTS),None,689,Theo Ratliff,1610612755.0,0,None,None,0,None,None
6,0020000001,6,1,11:29,3,12,None,None,None,MISS Ratliff Free Throw 2 of 2,None,689,Theo Ratliff,1610612755.0,0,None,None,0,None,None
7,0020000001,7,1,11:28,4,0,None,None,Ward REBOUND (Off:0 Def:1),None,None,369,Charlie Ward,1610612752.0,0,None,None,0,None,None
8,0020000001,8,1,11:18,6,2,None,None,None,Ratliff S.FOUL (P1.T1),None,689,Theo Ratliff,1610612755.0,0,None,None,0,None,None
9,0020000001,9,1,11:18,3,11,1 - 1,TIE,Camby Free Throw 1 of 2 (1 PTS),None,None,948,Marcus Camby,1610612752.0,0,None,None,0,None,None


## Initial event stream interpretation 

At this stage, we want to visually inspect the first game sample and answer the following questions:

    -Does 'eventnum' appear to order events correctly?
    -Does 'pctimestring' represent time remaining in the period? 
    -Are scoring updates visible in the 'score' field? 
    -Do substituition appear in the description field? 
    -Are players associated with events through 'player1','player2', and 'player3' in a consistent way? 
    
These observations will guide the event-mapping logic used in later notebooks

## Review event type frequency

The next step is to understand the distribution of high-level event type codes. This helps identify the main classes of events recorded in the table and provides a first look at whichcodes may correspond to shots, rebounds, fouls. substitutions, and period makers.

In [12]:
event_type_counts=(
    sample_game["eventmsgtype"]
    .value_counts()
    .sort_index()
    .rename_axis("eventmsgtype")
    .reset_index(name="count")
)

event_type_counts

,eventmsgtype,count
0,1,63
1,2,73
2,3,54
3,4,86
4,5,36
5,6,55
6,7,2
7,8,37
8,9,12
9,10,3


In [13]:
action_type_counts = (
    sample_game["eventmsgactiontype"]
    .value_counts()
    .sort_index()
    .rename_axis("eventmsgactiontype")
    .reset_index(name="count")
)

action_type_counts.head(20)

,eventmsgactiontype,count
0,0,127
1,1,145
2,2,29
3,3,6
4,4,12
5,5,33
6,6,1
7,8,4
8,10,3
9,11,26


## Multi-game event type validation

After inspecting one game, we also check event type frequencies across the full 50-game sample.

This helps confirm that the event categories used for substitutions, scoring, and period segmentation are consistently present across the broader sample.

In [14]:
multi_game_event_type_counts = (
    pbp_sample["eventmsgtype"]
    .value_counts()
    .sort_index()
    .rename_axis("eventmsgtype")
    .reset_index(name="count")
)

multi_game_event_type_counts

,eventmsgtype,count
0,1,35031
1,2,44847
2,3,25397
3,4,51292
4,5,15530
5,6,24068
6,7,1308
7,8,18414
8,9,6694
9,10,861


## Why event type codes matter 

The 'eventmsgtype' field will likely become one of the main tools for identifying event categories programatically. 

However, event codes should not be interpreted blindly. They should be cross-validated against the text descriptions and player fields to ensure the mapping is correct. 

For that reason, the next sections inspect event descriptions directly.

## Explore description fields

The play_by_play table includes three text description fields:

    -'homedescription'
    -'visitordescription'
    -'neutraldescription'
    
Together, these should help identify:

    -made and missed shots
    -fouls
    -rebounds
    -substitutions
    -period starts and ends
    -jump balls 
    -timeouts
    
We begin by checking how often each description field is populated.



In [15]:
description_summary=pd.DataFrame({
    "column":["homedescription","visitordescription","neutraldescription"],
    "non_null_count":[
        sample_game["homedescription"].notna().sum(),
        sample_game["visitordescription"].notna().sum(),
        sample_game["neutraldescription"].notna().sum()
    ]
})

description_summary

,column,non_null_count
0,homedescription,225
1,visitordescription,219
2,neutraldescription,10


In [16]:
sample_game[
    ["eventnum", "period", "pctimestring", "homedescription", "visitordescription", "neutraldescription", "score"]
].head(40)

,eventnum,period,pctimestring,homedescription,visitordescription,neutraldescription,score
0,0,1,12:00,None,None,Start of 1st Period (12:13 PM EST),None
1,1,1,12:00,Jump Ball Camby vs. Ratliff: Tip to Houston,None,None,None
2,2,1,11:41,MISS Sprewell 6' Jump Shot,Ratliff BLOCK (1 BLK),None,None
3,3,1,11:40,None,76ers Rebound,None,None
4,4,1,11:29,Camby S.FOUL (P1.T1),None,None,None
5,5,1,11:29,None,Ratliff Free Throw 1 of 2 (1 PTS),None,1 - 0
6,6,1,11:29,None,MISS Ratliff Free Throw 2 of 2,None,None
7,7,1,11:28,Ward REBOUND (Off:0 Def:1),None,None,None
8,8,1,11:18,None,Ratliff S.FOUL (P1.T1),None,None
9,9,1,11:18,Camby Free Throw 1 of 2 (1 PTS),None,None,1 - 1


## Multi-game substitution validation

We validate that substitution events are consistently represented across the multi-game sample using `eventmsgtype = 8`.

In [18]:
multi_game_sub_events = pbp_sample[pbp_sample["eventmsgtype"] == 8].copy()

multi_game_sub_events[sub_core_cols].head(20)

,game_id,eventnum,period,pctimestring,homedescription,visitordescription,neutraldescription,score
66,0020000001,67,1,4:16,None,SUB: McKie FOR Hill,None,None
68,0020000001,69,1,4:16,SUB: Childs FOR Ward,None,None,None
73,0020000001,74,1,3:52,SUB: Rice FOR Sprewell,None,None,None
79,0020000001,80,1,3:16,SUB: Thomas FOR Camby,None,None,None
88,0020000001,90,1,1:51,None,SUB: Maxwell FOR McKie,None,None
89,0020000001,91,1,1:51,None,SUB: MacCulloch FOR Ratliff,None,None
92,0020000001,94,1,1:49,None,SUB: McKie FOR Snow,None,None
124,0020000001,127,2,10:48,None,SUB: Snow FOR Iverson,None,None
134,0020000001,137,2,9:49,None,SUB: Iverson FOR Maxwell,None,None
140,0020000001,144,2,8:53,None,SUB: Maxwell FOR Snow,None,None


In [19]:
sub_core_cols = [
    "game_id",
    "eventnum",
    "period",
    "pctimestring",
    "homedescription",
    "visitordescription",
    "neutraldescription",
    "score"
]

multi_game_sub_events[sub_core_cols].head(20)

,game_id,eventnum,period,pctimestring,homedescription,visitordescription,neutraldescription,score
66,0020000001,67,1,4:16,None,SUB: McKie FOR Hill,None,None
68,0020000001,69,1,4:16,SUB: Childs FOR Ward,None,None,None
73,0020000001,74,1,3:52,SUB: Rice FOR Sprewell,None,None,None
79,0020000001,80,1,3:16,SUB: Thomas FOR Camby,None,None,None
88,0020000001,90,1,1:51,None,SUB: Maxwell FOR McKie,None,None
89,0020000001,91,1,1:51,None,SUB: MacCulloch FOR Ratliff,None,None
92,0020000001,94,1,1:49,None,SUB: McKie FOR Snow,None,None
124,0020000001,127,2,10:48,None,SUB: Snow FOR Iverson,None,None
134,0020000001,137,2,9:49,None,SUB: Iverson FOR Maxwell,None,None
140,0020000001,144,2,8:53,None,SUB: Maxwell FOR Snow,None,None


In [20]:
multi_game_sub_summary = (
    multi_game_sub_events
    .groupby("game_id")
    .size()
    .reset_index(name="substitution_count")
)

multi_game_sub_summary.describe()

,substitution_count
count,500.000000
mean,36.828000
std,7.200146
min,15.000000
25%,32.000000
50%,36.000000
75%,41.000000
max,70.000000


In [21]:
multi_game_sub_events[[
    "eventnum",
    "period",
    "pctimestring",
    "homedescription",
    "visitordescription",
    "neutraldescription",
    "player1_id",
    "player1_name",
    "player1_team_id",
    "player2_id",
    "player2_name",
    "player2_team_id",
    "player3_id",
    "player3_name",
    "player3_team_id"
]].head(20)

,eventnum,period,pctimestring,homedescription,visitordescription,neutraldescription,player1_id,player1_name,player1_team_id,player2_id,player2_name,player2_team_id,player3_id,player3_name,player3_team_id
66,67,1,4:16,None,SUB: McKie FOR Hill,None,238,Tyrone Hill,1610612755.0,243,Aaron McKie,1610612755.0,0,None,None
68,69,1,4:16,SUB: Childs FOR Ward,None,None,369,Charlie Ward,1610612752.0,164,Chris Childs,1610612752.0,0,None,None
73,74,1,3:52,SUB: Rice FOR Sprewell,None,None,84,Latrell Sprewell,1610612752.0,779,Glen Rice,1610612752.0,0,None,None
79,80,1,3:16,SUB: Thomas FOR Camby,None,None,948,Marcus Camby,1610612752.0,703,Kurt Thomas,1610612752.0,0,None,None
88,90,1,1:51,None,SUB: Maxwell FOR McKie,None,243,Aaron McKie,1610612755.0,137,Vernon Maxwell,1610612755.0,0,None,None
89,91,1,1:51,None,SUB: MacCulloch FOR Ratliff,None,689,Theo Ratliff,1610612755.0,1928,Todd MacCulloch,1610612755.0,0,None,None
92,94,1,1:49,None,SUB: McKie FOR Snow,None,727,Eric Snow,1610612755.0,243,Aaron McKie,1610612755.0,0,None,None
124,127,2,10:48,None,SUB: Snow FOR Iverson,None,947,Allen Iverson,1610612755.0,727,Eric Snow,1610612755.0,0,None,None
134,137,2,9:49,None,SUB: Iverson FOR Maxwell,None,137,Vernon Maxwell,1610612755.0,947,Allen Iverson,1610612755.0,0,None,None
140,144,2,8:53,None,SUB: Maxwell FOR Snow,None,727,Eric Snow,1610612755.0,137,Vernon Maxwell,1610612755.0,0,None,None


In [22]:
sub_clean = multi_game_sub_events.copy()

sub_clean["player_out"] = sub_clean["player1_name"]
sub_clean["player_in"] = sub_clean["player2_name"]
sub_clean["team_id"] = sub_clean["player1_team_id"]

sub_clean[[
    "game_id",
    "eventnum",
    "period",
    "pctimestring",
    "team_id",
    "player_out",
    "player_in"
]].head(10) 
    


,game_id,eventnum,period,pctimestring,team_id,player_out,player_in
66,0020000001,67,1,4:16,1610612755.0,Tyrone Hill,Aaron McKie
68,0020000001,69,1,4:16,1610612752.0,Charlie Ward,Chris Childs
73,0020000001,74,1,3:52,1610612752.0,Latrell Sprewell,Glen Rice
79,0020000001,80,1,3:16,1610612752.0,Marcus Camby,Kurt Thomas
88,0020000001,90,1,1:51,1610612755.0,Aaron McKie,Vernon Maxwell
89,0020000001,91,1,1:51,1610612755.0,Theo Ratliff,Todd MacCulloch
92,0020000001,94,1,1:49,1610612755.0,Eric Snow,Aaron McKie
124,0020000001,127,2,10:48,1610612755.0,Allen Iverson,Eric Snow
134,0020000001,137,2,9:49,1610612755.0,Vernon Maxwell,Allen Iverson
140,0020000001,144,2,8:53,1610612755.0,Eric Snow,Vernon Maxwell


### Interpretation

Substitution events are clearly represented using `eventmsgtype = 8`, and the player roles are consistently captured through `player1` (out) and `player2` (in).

This confirms that lineup transitions can be reconstructed programmatically without relying on text parsing.


## Substitution rule definition

Based on inspection of substitution events:

- `eventmsgtype = 8` identifies substitution events
- `player1_name` corresponds to the player leaving the game
- `player2_name` corresponds to the player entering the game
- `player1_team_id` identifies the team making the substitution

This allows us to construct structured substitution events without relying on text parsing.

This mapping will be used in Notebook 03 to update the on-court lineup state over time.

## Search for scoring events

To build stint-level point differential later, we need to understand how scoring events appear in the event stream. This section reviews rows with non-nul scores and compares them with event descriptions.

In [23]:
score_events = pbp_sample[pbp_sample["score"].notna()].copy()

score_events[[
    "game_id",
    "eventnum",
    "period",
    "pctimestring",
    "eventmsgtype",
    "eventmsgactiontype",
    "homedescription",
    "visitordescription",
    "neutraldescription",
    "score",
    "scoremargin"
]].head(30)

,game_id,eventnum,period,pctimestring,eventmsgtype,eventmsgactiontype,homedescription,visitordescription,neutraldescription,score,scoremargin
5,0020000001,5,1,11:29,3,11,None,Ratliff Free Throw 1 of 2 (1 PTS),None,1 - 0,-1
9,0020000001,9,1,11:18,3,11,Camby Free Throw 1 of 2 (1 PTS),None,None,1 - 1,TIE
10,0020000001,10,1,11:18,3,12,Camby Free Throw 2 of 2 (2 PTS),None,None,1 - 2,1
20,0020000001,20,1,10:08,1,5,None,Ratliff Layup (3 PTS) (Lynch 1 AST),None,3 - 2,-1
21,0020000001,21,1,9:50,1,1,Houston 16' Jump Shot (2 PTS) (Ward 1 AST),None,None,3 - 4,1
27,0020000001,27,1,8:56,1,8,None,Ratliff Slam Dunk (5 PTS) (Lynch 2 AST),None,5 - 4,-1
30,0020000001,30,1,8:31,1,42,None,Snow Driving Layup (2 PTS) (Iverson 1 AST),None,7 - 4,-3
31,0020000001,31,1,8:16,1,1,Houston 24' 3PT Jump Shot (5 PTS) (Johnson 1 AST),None,None,7 - 7,TIE
32,0020000001,32,1,7:57,1,1,None,Iverson 21' Jump Shot (2 PTS) (Snow 1 AST),None,9 - 7,-2
33,0020000001,33,1,7:40,1,1,Houston 17' Jump Shot (7 PTS),None,None,9 - 9,TIE


## Score progression interpretation

The `score` field appears to provide direct score updates during the game.

This is useful because it means point differential can be tracked without fully reconstructing every made shot from scratch.

In later notebooks, this field can be used to compute score change across stints by comparing the score at the start and end of each lineup segment.

## Inspect player roles across events

Because the table includes `player1`, `player2`, and `player3`, it is important to understand how these fields behave for different event categories.

For example:
- a made shot may involve a shooter and an assister
- a foul may involve the fouling player and the fouled player
- a substitution may involve an entering and exiting player

This section provides examples for manual inspection.

In [24]:
sample_game[[
    "eventnum",
    "period",
    "pctimestring",
    "eventmsgtype",
    "eventmsgactiontype",
    "homedescription",
    "visitordescription",
    "neutraldescription",
    "player1_name",
    "player2_name",
    "player3_name",
    "player1_team_id",
    "player2_team_id",
    "player3_team_id",
    "score"
]].head(50)

,eventnum,period,pctimestring,eventmsgtype,eventmsgactiontype,homedescription,visitordescription,neutraldescription,player1_name,player2_name,player3_name,player1_team_id,player2_team_id,player3_team_id,score
0,0,1,12:00,12,0,None,None,Start of 1st Period (12:13 PM EST),None,None,None,None,None,None,None
1,1,1,12:00,10,0,Jump Ball Camby vs. Ratliff: Tip to Houston,None,None,Marcus Camby,Theo Ratliff,Allan Houston,1610612752.0,1610612755.0,1610612752.0,None
2,2,1,11:41,2,1,MISS Sprewell 6' Jump Shot,Ratliff BLOCK (1 BLK),None,Latrell Sprewell,None,Theo Ratliff,1610612752.0,None,1610612755.0,None
3,3,1,11:40,4,0,None,76ers Rebound,None,None,None,None,None,None,None,None
4,4,1,11:29,6,2,Camby S.FOUL (P1.T1),None,None,Marcus Camby,None,None,1610612752.0,None,None,None
5,5,1,11:29,3,11,None,Ratliff Free Throw 1 of 2 (1 PTS),None,Theo Ratliff,None,None,1610612755.0,None,None,1 - 0
6,6,1,11:29,3,12,None,MISS Ratliff Free Throw 2 of 2,None,Theo Ratliff,None,None,1610612755.0,None,None,None
7,7,1,11:28,4,0,Ward REBOUND (Off:0 Def:1),None,None,Charlie Ward,None,None,1610612752.0,None,None,None
8,8,1,11:18,6,2,None,Ratliff S.FOUL (P1.T1),None,Theo Ratliff,None,None,1610612755.0,None,None,None
9,9,1,11:18,3,11,Camby Free Throw 1 of 2 (1 PTS),None,None,Marcus Camby,None,None,1610612752.0,None,None,1 - 1


## Create a preliminary event mapping table

At this stage, we can begin documenting a preliminary mapping between event types and basketball actions.

This mapping does not need to be perfect yet. The goal is to create a first reference that will later support event cleaning and possession logic.

In [25]:
event_mapping = pd.DataFrame({
    "eventmsgtype": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13],
    "event_category": [
        "Made Shot",
        "Missed Shot",
        "Free Throw",
        "Rebound",
        "Turnover",
        "Foul",
        "Violation",
        "Substitution",
        "Timeout",
        "Jump Ball",
        "Start Period",
        "End Period"
    ],
    "used_in_pipeline": [
        "Scoring and possessions",
        "Possessions",
        "Scoring and possessions",
        "Possessions",
        "Possessions",
        "Context and foul logic",
        "Context",
        "Structured substitution extraction for lineup reconstruction",
        "Context",
        "Game state initialization",
        "Period segmentation",
        "Period segmentation"
    ]
})

event_mapping

,eventmsgtype,event_category,used_in_pipeline
0,1,Made Shot,Scoring and possessions
1,2,Missed Shot,Possessions
2,3,Free Throw,Scoring and possessions
3,4,Rebound,Possessions
4,5,Turnover,Possessions
5,6,Foul,Context and foul logic
6,7,Violation,Context
7,8,Substitution,Structured substitution extraction for lineup reconstruction
8,9,Timeout,Context
9,10,Jump Ball,Game state initialization


## Define fields required for stint construction

The main deliverable of this notebook is a validated set of fields that will be used in Notebook 03.

For stint construction, the project needs:
- `game_id`
- `eventnum`
- `period`
- `pctimestring`
- `eventmsgtype`
- event descriptions
- score fields
- player/team reference fields

In [26]:
required_fields_check = pd.DataFrame({
    "required_field_role": [
        "Game identifier",
        "Event order",
        "Period",
        "Period clock",
        "Main event type",
        "Detailed action type",
        "Home description",
        "Visitor description",
        "Neutral description",
        "Score",
        "Score margin",
        "Primary player id",
        "Secondary player id",
        "Tertiary player id",
        "Primary team id"
    ],
    "candidate_column": [
        "game_id",
        "eventnum",
        "period",
        "pctimestring",
        "eventmsgtype",
        "eventmsgactiontype",
        "homedescription",
        "visitordescription",
        "neutraldescription",
        "score",
        "scoremargin",
        "player1_id",
        "player2_id",
        "player3_id",
        "player1_team_id"
    ],
    "exists_in_table": [
        "game_id" in sample_game.columns,
        "eventnum" in sample_game.columns,
        "period" in sample_game.columns,
        "pctimestring" in sample_game.columns,
        "eventmsgtype" in sample_game.columns,
        "eventmsgactiontype" in sample_game.columns,
        "homedescription" in sample_game.columns,
        "visitordescription" in sample_game.columns,
        "neutraldescription" in sample_game.columns,
        "score" in sample_game.columns,
        "scoremargin" in sample_game.columns,
        "player1_id" in sample_game.columns,
        "player2_id" in sample_game.columns,
        "player3_id" in sample_game.columns,
        "player1_team_id" in sample_game.columns
    ]
})

required_fields_check

,required_field_role,candidate_column,exists_in_table
0,Game identifier,game_id,True
1,Event order,eventnum,True
2,Period,period,True
3,Period clock,pctimestring,True
4,Main event type,eventmsgtype,True
5,Detailed action type,eventmsgactiontype,True
6,Home description,homedescription,True
7,Visitor description,visitordescription,True
8,Neutral description,neutraldescription,True
9,Score,score,True


In [27]:
sub_clean.to_csv("../data/interim/substitutions_sample.csv", index=False)

## Key findings

The `play_by_play` table contains the core fields needed to support event-level reconstruction for lineup analytics across multiple games.

The multi-game sample confirms that the dataset includes:

- consistent game identifiers
- event ordering through `eventnum`
- period and clock information
- score updates
- structured substitution events through `eventmsgtype = 8`
- player and team references through `player1`, `player2`, and `player3` fields

The single-game inspection remains useful for understanding event mechanics, while the multi-game checks confirm that the same logic can be generalized beyond the original prototype.

## Next step

Notebook 03 will move from event exploration to scalable stint reconstruction.

The next objective is to generalize the one-game stint engine so it can:

- process each game independently
- infer teams and starting lineups automatically
- track substitutions over time
- validate lineup integrity
- combine valid game-level stints into one multi-game dataset

This will create a larger and more useful dataset for lineup modeling and prediction.